# Renk Maskesi Kutu Değerlendirmesi ve TP / FP / FN — Murathan

**Hazır dedektör kullanılmamıştır** (yönerge 6. madde isteğe bağlıdır). Renk maskesinin bağlı bileşenleri aday kutu olur ve elle çizilmiş gerçek baret kutularıyla **IoU ≥ 0.50** bire bir eşleştirilir: eşleşme **TP**, eşleşmeyen aday **FP**, bulunamayan baret **FN**. Kutu = "sarı renkli bölge"; kutunun bulunması baretin başta olduğunu göstermez. Confidence değeri yoktur.

Fırçayla boyanmış gerçek baret maskesi varsa ek olarak **piksel kesinliği** (renkli piksellerin yüzde kaçı gerçekten baret — H1) ve **piksel duyarlılığı** (baret piksellerinin yüzde kaçı yakalandı) ölçülür.

In [ ]:
from pathlib import Path
import sys
KOK = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kaynak" / "analiz.py").is_file())
sys.path.insert(0, str(KOK))

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from kaynak.veri import ayarlari_oku, veri_setini_oku, goruntu_oku, maske_oku
from kaynak.analiz import analiz_et
from kaynak.gorsel import bgr2rgb, kutulari_ciz, maske_bindir
ayarlar = ayarlari_oku()
kayitlar = veri_setini_oku()
pd.set_option("display.max_colwidth", 80)
print("Görüntüler:", ", ".join(f"{k['kimlik']} ({k['bolum']})" for k in kayitlar))

## 1. Dondurulmuş ayarlar

In [ ]:
print(ayarlar["kutu_degerlendirme"])

## 2. Deney öncesi tahmin

In [ ]:
tahminler = pd.read_csv(KOK / "belgeler/tahminler.csv")
benim = tahminler[(tahminler.uye == "Murathan") & tahminler.deney_kimligi.str.startswith("color_mask")]
if benim.empty:
    print("UYARI: Bu deney için kayıtlı tahmin yok. Hoca: 'Tahmini olmayan deney puanlanmaz.'")
    print("Deneyi çalıştırmadan ÖNCE belgeler/tahminler.csv dosyasına iki cümle ekleyin.")
for _, t in benim.iterrows():
    print(f"[{t.kayit_tarihi}] {t.deney_kimligi}\n 1) {t.tahmin_1}\n 2) {t.tahmin_2}")

## 3. IoU ve eşleştirme kuralının elle kontrolü

In [ ]:
from kaynak.olcum import iou, kutulari_degerlendir
gt = [981, 94, 1070, 165]
print("IoU(kendisi) =", iou(gt, gt))
d = kutulari_degerlendir([gt, [985, 97, 1072, 168], [10, 10, 60, 60]], [gt])
print("Aynı barete iki kutu + bir alakasız kutu -> TP/FP/FN =", d["tp"], d["fp"], d["fn"])

## 4. Geliştirme ve kontrol sonuçları

In [ ]:
t = pd.read_csv(KOK / "sonuclar/temel_sonuclar.csv")
t[["kimlik", "bolum", "yontem", "aday_kutu", "tp", "fp", "fn", "eslesen_iou", "piksel_kesinlik_yuzde", "piksel_duyarlilik_yuzde"]]

In [ ]:
k = t[t.bolum == "kontrol"].groupby("yontem")[["tp", "fp", "fn"]].sum()
k["gercek_baret"] = sum(len(r["baret_kutulari"]) for r in veri_setini_oku("kontrol")); k

## 5. Görseller
Yeşil: gerçek baret · Mavi: TP · Kırmızı: FP · Gri: baş ROI

In [ ]:
fig, eksen = plt.subplots(4, 2, figsize=(14, 20))
for satir, kimlik in zip(eksen, ("K1", "K2", "K3", "K4")):
    for e, uzay in zip(satir, ("rgb", "hsv")):
        e.imshow(bgr2rgb(goruntu_oku(KOK / f"sonuclar/kutular/{kimlik}_{uzay}.png")))
        r = t[(t.kimlik == kimlik) & (t.yontem == uzay.upper())].iloc[0]
        e.set_title(f"{kimlik} {uzay.upper()}  TP={r.tp} FP={r.fp} FN={r.fn}"); e.axis("off")
plt.tight_layout(); plt.show()

## 6. Yanlış alarm ve kaçırma — neden ve sonraki kontrol
*(Sonuçlar yeni veriyle üretildikten sonra güncellenecek: bir yanlış alarm, bir kaçırma; her biri için iki cümlelik neden ve sonraki kontrol.)*